In [ ]:
#주요 미국 경제지표
FRED_SERIES = {
    # 물가
    "cpi": "CPIAUCSL",
    "core_cpi": "CPILFESL",
    "pce": "PCEPI",
    "core_pce": "PCEPILFE",

    # 고용
    "unemployment_rate": "UNRATE",
    "nonfarm_payroll": "PAYEMS",

    # 경기 / 성장
    "real_gdp": "GDPC1",
    "industrial_production": "INDPRO",
    "retail_sales": "RSAFS",

    # 금리
    "fed_funds_rate": "DFF",
    "treasury_2y": "DGS2",
    "treasury_10y": "DGS10",
    "treasury_10y_2y_spread": "T10Y2Y",

    # 소비 심리
    "consumer_sentiment": "UMCSENT",
}

In [ ]:
import os
import requests
import pandas as pd


FRED_API_KEY = os.getenv('FRED_API_KEY')

FRED_API_URL = "https://api.stlouisfed.org/fred/series/observations"


FRED_SERIES = {
    # 물가
    "cpi": "CPIAUCSL",
    "core_cpi": "CPILFESL",
    "pce": "PCEPI",
    "core_pce": "PCEPILFE",

    # 고용
    "unemployment_rate": "UNRATE",
    "nonfarm_payroll": "PAYEMS",

    # 경기 / 성장
    "real_gdp": "GDPC1",
    "industrial_production": "INDPRO",
    "retail_sales": "RSAFS",

    # 금리
    "fed_funds_rate": "DFF",
    "treasury_2y": "DGS2",
    "treasury_10y": "DGS10",
    "treasury_10y_2y_spread": "T10Y2Y",

    # 소비 심리
    "consumer_sentiment": "UMCSENT",
}


def fetch_fred_series( #지표 한개 받는 함수
    series_id,
    start_date=None,
    end_date=None,
    api_key=FRED_API_KEY,
):
    params = {
        "series_id": series_id,
        "api_key": api_key,
        "file_type": "json",
    }

    if start_date is not None:
        params["observation_start"] = start_date

    if end_date is not None:
        params["observation_end"] = end_date

    response = requests.get(
        FRED_API_URL,
        params=params,
        timeout=30,
    )

    response.raise_for_status()

    data = response.json()

    df = pd.DataFrame(data["observations"])

    df = df[["date", "value"]].copy()

    df["date"] = pd.to_datetime(df["date"])

    df["value"] = pd.to_numeric(
        df["value"],
        errors="coerce"
    )

    return df


def fetch_fred_indicators( # 모든 지표를 받기 위한 함수
    series_dict,  #지표 목록을 dict로 넘겨 받음
    start_date=None,
    end_date=None,
):
    results = []

    for indicator_name, series_id in series_dict.items():

        print(
            f"Fetching: "
            f"{indicator_name} ({series_id})"
        )

        df = fetch_fred_series(
            series_id=series_id,
            start_date=start_date,
            end_date=end_date,
        )

        df["indicator"] = indicator_name
        df["series_id"] = series_id

        results.append(df)

    result_df = pd.concat(
        results,
        ignore_index=True
    )

    result_df = result_df[
        [
            "date",
            "indicator",
            "series_id",
            "value",
        ]
    ]

    return result_df

In [6]:
economic_df = fetch_fred_indicators(
    series_dict=FRED_SERIES,
    start_date="2025-01-01",
    end_date="2026-10-14",
)

economic_df

Fetching: cpi (CPIAUCSL)
Fetching: core_cpi (CPILFESL)
Fetching: pce (PCEPI)
Fetching: core_pce (PCEPILFE)
Fetching: unemployment_rate (UNRATE)
Fetching: nonfarm_payroll (PAYEMS)
Fetching: real_gdp (GDPC1)
Fetching: industrial_production (INDPRO)
Fetching: retail_sales (RSAFS)
Fetching: fed_funds_rate (DFF)
Fetching: treasury_2y (DGS2)
Fetching: treasury_10y (DGS10)
Fetching: treasury_10y_2y_spread (T10Y2Y)
Fetching: consumer_sentiment (UMCSENT)


,date,indicator,series_id,value
0,2025-01-01,cpi,CPIAUCSL,318.961
1,2025-02-01,cpi,CPIAUCSL,319.679
2,2025-03-01,cpi,CPIAUCSL,319.785
3,2025-04-01,cpi,CPIAUCSL,320.302
4,2025-05-01,cpi,CPIAUCSL,320.620
...,...,...,...,...
2216,2026-04-01,consumer_sentiment,UMCSENT,49.800
2217,2026-05-01,consumer_sentiment,UMCSENT,44.800
2218,2026-06-01,consumer_sentiment,UMCSENT,49.500
2219,2026-07-01,consumer_sentiment,UMCSENT,55.200


In [ ]:
economic_wide_df = (
    economic_df
    .pivot(
        index="date",
        columns="indicator",
        values="value",
    )
    .reset_index()
)

economic_wide_df # 지표별 주기가 다르기 때문에 NaN이 많이 발생
# 2026년 8월 1일에 그 CPI가 시장에 공개됐다는 의미가 아닙니다.
# 이 date는 기본적으로 해당 관측값이 어느 경제 기간에 해당하는지 나타내는 observation date입니다. 발표 시점 분석과는 별개의 개념

indicator,date,consumer_sentiment,core_cpi,core_pce,cpi,fed_funds_rate,industrial_production,nonfarm_payroll,pce,real_gdp,retail_sales,treasury_10y,treasury_10y_2y_spread,treasury_2y,unemployment_rate
0,2025-01-01,71.7,324.638,124.609,318.961,4.33,100.0647,158268.0,125.423,23650.213,687058.0,NaN,NaN,NaN,4.0
1,2025-01-02,NaN,NaN,NaN,NaN,4.33,NaN,NaN,NaN,NaN,NaN,4.57,0.32,4.25,NaN
2,2025-01-03,NaN,NaN,NaN,NaN,4.33,NaN,NaN,NaN,NaN,NaN,4.60,0.32,4.28,NaN
3,2025-01-04,NaN,NaN,NaN,NaN,4.33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-01-05,NaN,NaN,NaN,NaN,4.33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
642,2026-10-05,NaN,NaN,NaN,NaN,3.88,NaN,NaN,NaN,NaN,NaN,5.31,0.47,4.84,NaN
643,2026-10-06,NaN,NaN,NaN,NaN,3.88,NaN,NaN,NaN,NaN,NaN,5.27,0.48,4.79,NaN
644,2026-10-07,NaN,NaN,NaN,NaN,3.88,NaN,NaN,NaN,NaN,NaN,5.28,0.51,4.77,NaN
645,2026-10-08,NaN,NaN,NaN,NaN,3.88,NaN,NaN,NaN,NaN,NaN,5.22,0.47,4.75,NaN
